# 03 ECL Engine

Phase 3 builds a simplified IFRS 9-style Expected Credit Loss engine using existing Phase 2 PD predictions. This notebook does not retrain the PD model and does not build the dashboard.

## Project Context

The Credit Risk and IFRS 9 Expected Credit Loss Engine is a portfolio project that connects credit risk modeling, business interpretation, and risk analytics. Phase 3 converts PD scores into starter ECL estimates using transparent EAD, LGD, and simplified staging assumptions.

## IFRS 9 ECL Objective

Estimate row-level and portfolio-level expected credit loss using:

`ECL = PD x LGD x EAD`

The staging logic is a simplified portfolio analytics proxy and is not an official IFRS 9 compliance model.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FIGURES_DIR, OUTPUTS_DIR
from src.ecl import (
    assign_ead,
    assign_ifrs9_stage,
    assign_lgd,
    calculate_ecl,
    create_scenario_summary,
    load_pd_predictions,
    summarize_ecl,
)
from src.visualization import (
    plot_ecl_by_group,
    plot_ecl_distribution,
    plot_scenario_comparison,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUTS_DIR / "predictions").mkdir(parents=True, exist_ok=True)

## Load PD Predictions

In [ ]:
pd_predictions_path = OUTPUTS_DIR / "predictions" / "pd_predictions.csv"
predictions = load_pd_predictions(pd_predictions_path)
print(f"Loaded PD predictions: {pd_predictions_path}")
print(f"Shape: {predictions.shape}")
display(predictions.head())

## Review Available Fields

In [ ]:
required_fields = ["row_id", "default_flag", "pd_score", "pd_score_band"]
requested_original_fields = [
    "loan_amnt",
    "installment",
    "term",
    "int_rate",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "revol_util",
    "total_acc",
]
available_original_fields = [column for column in requested_original_fields if column in predictions.columns]
missing_original_fields = [column for column in requested_original_fields if column not in predictions.columns]

display(pd.DataFrame({"column": predictions.columns.tolist()}))
print(f"Available original fields for ECL output: {available_original_fields}")
print(f"Missing requested original fields: {missing_original_fields}")

## Define EAD Assumption

EAD uses `loan_amnt` when available. If unavailable, the fallback is `installment x 36`; if that is unavailable, the fallback is a fixed placeholder exposure of 10,000.

In [ ]:
ecl_df, ead_method = assign_ead(predictions)
print(f"EAD method used: {ead_method}")
display(ecl_df[available_original_fields + ["ead"]].head())

## Define LGD Assumption

Base LGD is 45%. If `home_ownership` is available, LGD is adjusted using a simple portfolio assumption: MORTGAGE = 35%, OWN = 40%, RENT = 50%, other or missing = 45%.

In [ ]:
ecl_df, lgd_method = assign_lgd(ecl_df)
print(f"LGD method used: {lgd_method}")
display(ecl_df[[column for column in ["home_ownership", "lgd"] if column in ecl_df.columns]].head())
display(ecl_df["lgd"].value_counts(dropna=False).sort_index().to_frame("row_count"))

## Define IFRS 9-style Staging Logic

- Stage 1: `pd_score < 0.20`
- Stage 2: `0.20 <= pd_score < 0.50`
- Stage 3: `pd_score >= 0.50` or `default_flag == 1`

This is simplified staging for portfolio analytics, not official IFRS 9 compliance.

In [ ]:
ecl_df = assign_ifrs9_stage(ecl_df)
display(ecl_df["ifrs9_stage"].value_counts().sort_index().to_frame("row_count"))

## Calculate Base Expected Credit Loss

In [ ]:
ecl_df = calculate_ecl(ecl_df)
base_total_exposure = ecl_df["ead"].sum()
base_total_ecl = ecl_df["ecl"].sum()
base_ecl_rate = base_total_ecl / base_total_exposure

print(f"Base total exposure: {base_total_exposure:,.2f}")
print(f"Base total ECL: {base_total_ecl:,.2f}")
print(f"Base ECL rate: {base_ecl_rate:.2%}")
display(ecl_df[["row_id", "default_flag", "pd_score", "pd_score_band", "ead", "lgd", "ifrs9_stage", "ecl"]].head())

image = plot_ecl_distribution(ecl_df, ecl_col="ecl")
image.save(FIGURES_DIR / "ecl_distribution.png")
display(image)

## Aggregate ECL by Stage

In [ ]:
stage_summary = summarize_ecl(ecl_df, "ifrs9_stage")
display(stage_summary)

image = plot_ecl_by_group(stage_summary, "ifrs9_stage", value_col="total_ecl", title="ECL by IFRS 9-style Stage")
image.save(FIGURES_DIR / "ecl_by_stage.png")
display(image)

image = plot_ecl_by_group(stage_summary, "ifrs9_stage", value_col="total_exposure", title="Exposure by IFRS 9-style Stage")
image.save(FIGURES_DIR / "exposure_by_stage.png")
display(image)

## Aggregate ECL by PD Score Band

In [ ]:
score_band_summary = summarize_ecl(ecl_df, "pd_score_band")
display(score_band_summary)

image = plot_ecl_by_group(score_band_summary, "pd_score_band", value_col="total_ecl", title="ECL by PD Score Band")
image.save(FIGURES_DIR / "ecl_by_score_band.png")
display(image)

## Aggregate ECL by Grade if Available

In [ ]:
if "grade" in ecl_df.columns:
    grade_summary = summarize_ecl(ecl_df, "grade")
    display(grade_summary)
    image = plot_ecl_by_group(grade_summary, "grade", value_col="total_ecl", title="ECL by Grade")
    image.save(FIGURES_DIR / "ecl_by_grade.png")
    display(image)
else:
    grade_summary = pd.DataFrame()
    print("Skipped grade ECL summary because grade is not available.")

## Aggregate ECL by Purpose if Available

In [ ]:
if "purpose" in ecl_df.columns:
    purpose_summary = summarize_ecl(ecl_df, "purpose")
    display(purpose_summary)
    image = plot_ecl_by_group(purpose_summary, "purpose", value_col="total_ecl", title="ECL by Purpose")
    image.save(FIGURES_DIR / "ecl_by_purpose.png")
    display(image)
else:
    purpose_summary = pd.DataFrame()
    print("Skipped purpose ECL summary because purpose is not available.")

## Scenario Analysis

In [ ]:
scenarios = [
    {"scenario_name": "Base", "pd_multiplier": 1.00, "lgd_multiplier": 1.00},
    {"scenario_name": "Mild stress", "pd_multiplier": 1.25, "lgd_multiplier": 1.10},
    {"scenario_name": "Severe stress", "pd_multiplier": 1.50, "lgd_multiplier": 1.20},
]
scenario_summary = create_scenario_summary(ecl_df, scenarios)
display(scenario_summary)

scenario_path = OUTPUTS_DIR / "predictions" / "ecl_scenario_summary.csv"
scenario_summary.to_csv(scenario_path, index=False)
print(f"Saved scenario summary: {scenario_path}")

image = plot_scenario_comparison(scenario_summary, value_col="total_ecl")
image.save(FIGURES_DIR / "scenario_ecl_comparison.png")
display(image)

## Business Interpretation

In [ ]:
top_stage = stage_summary.sort_values("total_ecl", ascending=False).iloc[0]
print(f"The largest ECL concentration by stage is {top_stage['ifrs9_stage']} with ECL of {top_stage['total_ecl']:,.2f}.")

if not grade_summary.empty:
    top_grade = grade_summary.sort_values("total_ecl", ascending=False).iloc[0]
    print(f"The largest ECL concentration by grade is {top_grade['grade']} with ECL of {top_grade['total_ecl']:,.2f}.")
if not purpose_summary.empty:
    top_purpose = purpose_summary.sort_values("total_ecl", ascending=False).iloc[0]
    print(f"The largest ECL concentration by purpose is {top_purpose['purpose']} with ECL of {top_purpose['total_ecl']:,.2f}.")

severe_ecl = scenario_summary.loc[scenario_summary["scenario"] == "Severe stress", "total_ecl"].iloc[0]
print(f"Severe stress increases total ECL to {severe_ecl:,.2f}, compared with base ECL of {base_total_ecl:,.2f}.")

## Limitations

- EAD is approximated from available portfolio fields and does not represent contractual exposure modeling.
- LGD is a simplified fixed or home-ownership-adjusted assumption, not a recovery model.
- IFRS 9-style staging is based only on PD score thresholds and default flag.
- Scenario analysis applies simple PD and LGD multipliers only.
- No macroeconomic scenario model, lifetime PD curve, discounting, cure logic, or regulatory validation is included.

## Outputs for Dashboard

In [ ]:
output_columns = [
    "row_id",
    "default_flag",
    "pd_score",
    "pd_score_band",
    "ead",
    "lgd",
    "ifrs9_stage",
    "ecl",
] + available_original_fields
output_columns = [column for column in output_columns if column in ecl_df.columns]
ecl_results = ecl_df[output_columns].copy()
ecl_results_path = OUTPUTS_DIR / "ecl_results.csv"
ecl_results.to_csv(ecl_results_path, index=False)

print(f"Saved ECL results: {ecl_results_path}")
print(f"ECL results shape: {ecl_results.shape}")
display(ecl_results.head())

## Next Steps

- Build the business interpretation notebook using the ECL outputs.
- Build the Streamlit dashboard after the business interpretation layer is documented.
- Consider improving EAD, LGD, staging, and scenario assumptions before presenting the project as an IFRS 9-ready model.